## Logistic Regression

Load up required libraries and dataset. Convert the data to a dataframe containing a subset of the data:

In [ ]:
import pandas as pd
import numpy as np
pumpkins = pd.read_csv("US-pumpkins.csv")


In [ ]:
# Select the columns we want to use
pumpkins = pumpkins[['Item Size', 'Variety', 'Package', 'Color']]

# Drop rows with missing values
pumpkins = pumpkins.dropna()



# Let's have a look to our data!

By visualising it with Seaborn

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Specify colors for each values of the hue variable
palette = {
    'ORANGE': 'orange',
    'WHITE': 'wheat'
}

# Plot a bar plot to visualize how many pumpkins of each variety are orange or white
plt.figure(figsize=(10,6))
sns.countplot(data=pumpkins,
              x='Variety',
              hue='Color',
              palette=palette)

plt.xticks(rotation=90)
plt.show()

# Data pre-processing

Let's encode features and labels to better plot the data and train the model

In [ ]:
# Let's look at the different values of the 'Item Size' column
pumpkins['Item Size'].unique()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
# Encode the 'Item Size' column using ordinal encoding
ordinal_features = ['Item Size']

ordinal_encoder = OrdinalEncoder()

pumpkins[ordinal_features] = ordinal_encoder.fit_transform(
    pumpkins[ordinal_features]
)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
# Encode all the other features using one-hot encoding
categorical_features = ['Variety', 'Package']

categorical_encoder = OneHotEncoder(handle_unknown='ignore')

In [ ]:
from sklearn.compose import ColumnTransformer
ct = ColumnTransformer(transformers=[
     ('ord', ordinal_encoder, ordinal_features),
     ('cat', categorical_encoder, categorical_features)
     ])
# Get the encoded features as a pandas DataFrame
encoded_features = ct.fit_transform(pumpkins)

encoded_pumpkins = pd.DataFrame(
    encoded_features.toarray(),
    columns=ct.get_feature_names_out()
)

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Encode the 'Color' column using label encoding
label_encoder = LabelEncoder()

encoded_pumpkins["Color"] = label_encoder.fit_transform(
    pumpkins["Color"]
)

In [ ]:
# Let's look at the mapping between the encoded values and the original values
mapping = dict(zip(label_encoder.classes_,
                   label_encoder.transform(label_encoder.classes_)))

print(mapping)

# Analysing relationships between features and label

In [ ]:
palette = {
    'ORANGE': 'orange',
    'WHITE': 'wheat',
}
# We need the encoded Item Size column to use it as the x-axis values in the plot

# Defining axis labels
sns.scatterplot(
    data=pumpkins,
    x='Item Size',
    y='Variety',
    hue='Color',
    palette=palette
)

plt.show()

Let's now focus on a specific relationship: Item Size and Color!

In [ ]:
import warnings
warnings.filterwarnings(action='ignore', category=UserWarning, module='seaborn')
palette = {
    0: 'orange',
    1: 'wheat'
}

sns.swarmplot(
    x="Color",
    y="ord__Item Size",
    hue="Color",
    data=encoded_pumpkins,
    palette=palette
)

plt.show()

In [ ]:
# Suppressing warning message claiming that a portion of points cannot be placed into the plot due to the high number of data points
import warnings
warnings.filterwarnings(action='ignore', category=UserWarning, module='seaborn')

palette = {
    0: 'orange',
    1: 'wheat'
}
sns.swarmplot(x="Color", y="ord__Item Size", hue="Color", data=encoded_pumpkins, palette=palette)

**Watch out**: Ignoring warnings is NOT a best practice and should be avoid, whenever possible. Warnings often contain useful messages that let us improve our code and solve an issue.
The reason why we are ignoring this specific warning is to guarantee the readability of the plot. Plotting all the data points with a reduced marker size, while keeping consistency with the palette color, generates an unclear visualization.

# Build your model

In [ ]:
from sklearn.model_selection import train_test_split
# X is the encoded features
X = encoded_pumpkins.drop("Color", axis=1)
# y is the encoded label
y = encoded_pumpkins["Color"]
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression

# Train a logistic regression model on the pumpkin dataset
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)
# Evaluate the model and print the results
predictions = model.predict(X_test)

print("F1 Score:")
print(f1_score(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions))

In [ ]:
from sklearn.metrics import confusion_matrix
#
cm = confusion_matrix(y_test, predictions)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()